# Turtle Breakout 08 — risk per trade from 1% to 100% on BTC

Pre-registration: `discussions/Turtle_Breakout/010_claude_tb08_preregistration.md`. Code: `src/data_engineering/modeling/turtle_sizing.py`.

TB 01 daily long-only turtle on the BTCUSDT perpetual; only the risk fraction changes: $u = f \cdot \mathrm{HWM} / (2\,\mathrm{ATR})$, $f = 1\%, 2\%, \dots, 100\%$.
**Primary sweep P:** cap 125× notional, sizing on the high-water mark. **Reported:** C3 (TB 01's 3× cap), EQ (sizing on current equity).
**Decision:** for each $f \ne 2\%$, the paired difference in mean daily log return vs 2%, 21-day block bootstrap, Holm over 99 at $\alpha = 0.05$.

Environment: `TB01_DATA_DIR` (default `BINANCE_UM_DIR\long`), `TB08_SYNTHETIC=1`.

In [ ]:
import os, sys, json, subprocess, tempfile, warnings, platform
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 120)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_trade_features as tf, turtle_sizing as ts

SYNTHETIC = os.getenv('TB08_SYNTHETIC', '0') == '1'
P = tb.Params(); SP = ts.SParams()
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb08-')) / '08'; SP = ts.SParams(n_boot=500, n_kelly_boot=500)
    b = tb.synthetic_5m(2400, seed=11, trend=0.01, start='2020-01-01'); f = tb.synthetic_funding(b, 0.0001)
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '08'
    base = Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance'
    DATA = Path(os.environ.get('TB01_DATA_DIR', str(base / 'long')))
    b, _ = tf.load_um(DATA / 'BTCUSDT_um_1m.parquet')
    f = tb.funding_events(pd.read_parquet(DATA / 'basis' / 'funding' / 'BTCUSDT_funding.parquet'))
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_turtle_sizing.py'], cwd=str(ROOT), env=env, capture_output=True, text=True)
print(chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
D = tb.prepare_asset('BTC', b, f, P)
START = b.index[0] + pd.Timedelta(days=30)
print('BTC', b.index[0], '->', b.index[-1], '| start', START)
print(json.dumps(ts.registration(SP), indent=1))

## 1. Reproduction check: f = 2%, 3× cap = TB 01 single-asset BTC

In [ ]:
E, T, st = ts.run_f(D, P, 0.02, START, 3.0)
m = tb.metrics(E, T); print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items()})
if not SYNTHETIC:
    ref = pd.read_csv(ROOT / 'notebooks' / 'Turtle_Breakout' / '01' / '01_single_asset.csv')
    display(ref[ref.iloc[:, 0] == 'BTC'])

## 2. The three sweeps

In [ ]:
TAB, CUR = {}, {}
for name, cap, sb in (('P', SP.max_lev_primary, 'hwm'), ('C3', SP.max_lev_c3, 'hwm'), ('EQ', SP.max_lev_primary, 'equity')):
    TAB[name], CUR[name] = ts.sweep(D, P, START, cap, sb)
    print(name, 'done')
cols = ['f', 'cagr', 'log_growth_per_year', 'sharpe', 'max_dd', 'calmar', 'ruined', 'worst_trade_pct_equity', 'mean_gross_lev', 'max_gross_lev', 'capped_entries', 'final_equity']
sel = [0.01, 0.02, 0.03, 0.05, 0.075, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.75, 1.0]
for name in TAB:
    print(name); display(TAB[name][TAB[name].f.round(3).isin([round(x, 3) for x in sel])][cols].round(3))

## 3. Decision: growth vs 2% (P sweep, Holm over 99)

In [ ]:
CMP = ts.compare(CUR['P'], SP)
DEC = ts.decide(CMP, TAB['P'])
display(CMP[CMP.f.isin(sel)].round(4))
print(json.dumps(DEC, indent=1, default=float))
CMP_C3 = ts.compare(CUR['C3'], SP); CMP_EQ = ts.compare(CUR['EQ'], SP)
DEC_C3 = ts.decide(CMP_C3, TAB['C3']); DEC_EQ = ts.decide(CMP_EQ, TAB['EQ'])
print('C3:', {k: v for k, v in DEC_C3.items() if 'than' not in k}); print('EQ:', {k: v for k, v in DEC_EQ.items() if 'than' not in k})

## 4. Kelly: growth-optimal f on the trades, and how uncertain it is (reported)

In [ ]:
E2, T2, _ = ts.run_f(D, P, 0.02, START, SP.max_lev_primary)
k = tb.period_id(pd.DatetimeIndex(T2.entry_time), 'day') - D.pid0
R = (T2.ret_notional / (P.stop_atr * D.atr[k] / T2.entry_px)).to_numpy()
R = R[T2.reason.to_numpy() != 'end']
KC = ts.kelly_curve(R); KB = ts.kelly_boot(R, SP)
KELLY = {'trades': int(len(R)), 'mean_R': float(R.mean()), 'win_rate': float((R > 0).mean()), 'worst_R': float(R.min()),
         'kelly_f': float(KC.idxmax()), 'half_kelly_f': float(KC.idxmax() / 2),
         'boot_q05': float(np.quantile(KB, .05)), 'boot_median': float(np.median(KB)), 'boot_q95': float(np.quantile(KB, .95)),
         'boot_share_f_le_2pct': float((KB <= 0.02).mean())}
print(json.dumps(KELLY, indent=1))

## 5. Figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for name, col in (('P', '#1f5aa6'), ('C3', '#c2571a'), ('EQ', '#3a7d44')):
    t = TAB[name]
    axes[0, 0].plot(t.f * 100, t.log_growth_per_year.clip(lower=-1), color=col, label=name)
    axes[0, 1].plot(t.f * 100, t.max_dd * 100, color=col, label=name)
    axes[1, 0].plot(t.f * 100, t.sharpe, color=col, label=name)
axes[0, 0].set_ylabel('log growth per year (clipped at -1)'); axes[0, 1].set_ylabel('max drawdown %'); axes[1, 0].set_ylabel('Sharpe (daily)')
c = CMP
axes[0, 0].scatter(c.f[c.growth_result == 'HIGHER_GROWTH'] * 100, [TAB['P'].set_index('f').log_growth_per_year[x] for x in c.f[c.growth_result == 'HIGHER_GROWTH']], color='k', s=10, zorder=4, label='P: significantly > 2%')
for a in axes.ravel()[:3]:
    a.axvline(2, color='k', lw=0.7, ls=':'); a.set_xlabel('risk per trade f (%)'); a.grid(alpha=0.25); a.legend(frameon=False, fontsize=8)
for x, col in zip((0.01, 0.02, 0.05, 0.1, KELLY['half_kelly_f'], KELLY['kelly_f']), ('0.6', 'k', '#1f5aa6', '#3a7d44', '#c2571a', '#b33f40')):
    x = round(min(max(x, 0.01), 1.0), 2)
    axes[1, 1].plot(CUR['P'].index, CUR['P'][x], color=col, lw=1.2, label=f'f = {x:.0%}')
axes[1, 1].set_yscale('log'); axes[1, 1].set_ylabel('equity (log)'); axes[1, 1].legend(frameon=False, fontsize=8); axes[1, 1].grid(alpha=0.25)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(KB * 100, bins=np.arange(0.5, 101.5, 1), color='#1f5aa6', alpha=0.7); ax.axvline(KELLY['kelly_f'] * 100, color='k', lw=1.5)
ax.axvline(2, color='#b33f40', lw=1, ls=':'); ax.set_xlabel('growth-optimal f (%) in trade bootstrap'); ax.set_title('Kelly optimum: point estimate (black) and bootstrap; 2% dotted', loc='left', fontsize=10)
plt.tight_layout(); plt.show()

## 6. Save

In [ ]:
for name in TAB:
    TAB[name].to_csv(OUT / f'08_sweep_{name}.csv', index=False)
CMP.to_csv(OUT / '08_compare_P.csv', index=False); CMP_C3.to_csv(OUT / '08_compare_C3.csv', index=False); CMP_EQ.to_csv(OUT / '08_compare_EQ.csv', index=False)
CUR['P'][[x for x in (0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0)]].to_csv(OUT / '08_curves_P_selected.csv')
(OUT / '08_decision.json').write_text(json.dumps({'decision': DEC, 'C3': DEC_C3, 'EQ': DEC_EQ, 'kelly': KELLY,
    'reproduction_2pct_cap3': m, 'registration': ts.registration(SP), 'synthetic': SYNTHETIC}, indent=1, default=str))
git = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(ROOT), capture_output=True, text=True).stdout.strip()
(OUT / '08_manifest.json').write_text(json.dumps({'version': ts.VERSION, 'tb01_version': tb.VERSION, 'git': git, 'python': platform.python_version(),
    'created_utc': pd.Timestamp.now('UTC').isoformat(), 'start': str(START), 'synthetic': SYNTHETIC}, indent=1))
print('saved to', OUT); print('VERDICT:', DEC['verdict'])